In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, TimestampType 

In [2]:
spark = SparkSession.builder.appName("transactions").master("local[*]").getOrCreate()

In [3]:
# ==========================================
# TABLA 1: USERS
# ==========================================

users_data = [
    Row(user_id=1, full_name="Carlos Fernandez", country="CO", signup_date="2021-03-15", risk_level="LOW"),
    Row(user_id=2, full_name="Maria Lopez",      country="CO", signup_date="2020-11-02", risk_level="LOW"),
    Row(user_id=3, full_name="John Smith",       country="US", signup_date="2022-01-10", risk_level="MEDIUM"),
    Row(user_id=4, full_name="Ana Torres",       country="CO", signup_date="2023-06-20", risk_level="LOW"),
    Row(user_id=5, full_name="Pedro Gomez",      country="MX", signup_date="2019-08-05", risk_level="HIGH"),
    Row(user_id=6, full_name="Laura Diaz",       country="CO", signup_date="2021-12-01", risk_level="LOW"),
    Row(user_id=7, full_name="Kevin Brown",      country="US", signup_date="2022-09-14", risk_level="MEDIUM"),
    Row(user_id=8, full_name="Sofia Ramirez",    country="CO", signup_date="2020-02-28", risk_level="LOW"),
    Row(user_id=9, full_name="Diego Castro",     country="MX", signup_date="2023-01-01", risk_level="HIGH"),
    Row(user_id=10, full_name="Julia Perez",     country="CO", signup_date="2021-07-07", risk_level="LOW"),
]

df_users = spark.createDataFrame(users_data)
df_users.createOrReplaceTempView("users")


# ==========================================
# TABLA 2: MERCHANTS
# ==========================================

merchants_data = [
    Row(merchant_id=1, merchant_name="Amazon",    category="E-commerce", merchant_country="US"),
    Row(merchant_id=2, merchant_name="Walmart",    category="Retail",     merchant_country="US"),
    Row(merchant_id=3, merchant_name="Netflix",    category="Streaming",  merchant_country="US"),
    Row(merchant_id=4, merchant_name="Uber",       category="Transport",  merchant_country="US"),
    Row(merchant_id=5, merchant_name="Falabella",  category="Retail",     merchant_country="CO"),
    Row(merchant_id=6, merchant_name="Rappi",      category="Delivery",   merchant_country="CO"),
    Row(merchant_id=7, merchant_name="Steam",      category="Gaming",     merchant_country="US"),
    Row(merchant_id=8, merchant_name="Exito",      category="Retail",     merchant_country="CO"),
]

df_merchants = spark.createDataFrame(merchants_data)
df_merchants.createOrReplaceTempView("merchants")


# ==========================================
# TABLA 3: CARDS
# ==========================================

cards_data = [
    Row(card_id=1,  user_id=1,  card_type="credit", issued_date="2021-04-01", is_blocked=False),
    Row(card_id=2,  user_id=1,  card_type="debit",  issued_date="2022-01-15", is_blocked=False),
    Row(card_id=3,  user_id=2,  card_type="credit", issued_date="2020-12-01", is_blocked=False),
    Row(card_id=4,  user_id=3,  card_type="credit", issued_date="2022-02-01", is_blocked=False),
    Row(card_id=5,  user_id=4,  card_type="debit",  issued_date="2023-07-01", is_blocked=False),
    Row(card_id=6,  user_id=5,  card_type="credit", issued_date="2019-09-01", is_blocked=True),
    Row(card_id=7,  user_id=6,  card_type="debit",  issued_date="2022-01-01", is_blocked=False),
    Row(card_id=8,  user_id=7,  card_type="credit", issued_date="2022-10-01", is_blocked=False),
    Row(card_id=9,  user_id=8,  card_type="credit", issued_date="2020-03-01", is_blocked=False),
    Row(card_id=10, user_id=9,  card_type="credit", issued_date="2023-02-01", is_blocked=False),
    Row(card_id=11, user_id=10, card_type="debit",  issued_date="2021-08-01", is_blocked=False),
    Row(card_id=12, user_id=2,  card_type="debit",  issued_date="2023-03-01", is_blocked=False),
]

df_cards = spark.createDataFrame(cards_data)
df_cards.createOrReplaceTempView("cards")


# ==========================================
# TABLA 4: TRANSACTIONS
# ==========================================

transactions_data = [
    Row(tx_id=1,  user_id=1, card_id=1,  merchant_id=1, amount=120.0, currency="USD", tx_date="2026-07-19 09:00:00", status="APPROVED"),
    Row(tx_id=2,  user_id=1, card_id=1,  merchant_id=2, amount=45.5,  currency="USD", tx_date="2026-07-19 14:30:00", status="APPROVED"),
    Row(tx_id=3,  user_id=1, card_id=2,  merchant_id=4, amount=22.0,  currency="USD", tx_date="2026-07-20 08:00:00", status="APPROVED"),
    Row(tx_id=4,  user_id=2, card_id=3,  merchant_id=5, amount=95.0,  currency="COP", tx_date="2026-07-19 10:00:00", status="APPROVED"),
    Row(tx_id=5,  user_id=2, card_id=3,  merchant_id=6, amount=22.3,  currency="COP", tx_date="2026-07-20 19:45:00", status="APPROVED"),
    Row(tx_id=6,  user_id=2, card_id=12, merchant_id=3, amount=15.0,  currency="USD", tx_date="2026-07-21 08:00:00", status="APPROVED"),
    Row(tx_id=7,  user_id=3, card_id=4,  merchant_id=4, amount=18.0,  currency="USD", tx_date="2026-07-19 08:15:00", status="APPROVED"),
    # --- ráfaga sospechosa (velocity) ---
    Row(tx_id=8,  user_id=3, card_id=4,  merchant_id=1, amount=50.0,  currency="USD", tx_date="2026-07-20 23:01:00", status="APPROVED"),
    Row(tx_id=9,  user_id=3, card_id=4,  merchant_id=1, amount=150.0, currency="USD", tx_date="2026-07-20 23:04:00", status="APPROVED"),
    Row(tx_id=10, user_id=3, card_id=4,  merchant_id=2, amount=300.0, currency="USD", tx_date="2026-07-20 23:06:00", status="APPROVED"),
    Row(tx_id=11, user_id=3, card_id=4,  merchant_id=7, amount=600.0, currency="USD", tx_date="2026-07-20 23:07:00", status="APPROVED"),
    Row(tx_id=12, user_id=3, card_id=4,  merchant_id=1, amount=999.0, currency="USD", tx_date="2026-07-20 23:09:00", status="DECLINED"),
    Row(tx_id=13, user_id=4, card_id=5,  merchant_id=6, amount=12.0,  currency="COP", tx_date="2026-07-19 12:00:00", status="APPROVED"),
    Row(tx_id=14, user_id=4, card_id=5,  merchant_id=8, amount=60.0,  currency="COP", tx_date="2026-07-20 18:00:00", status="APPROVED"),
    # --- tarjeta bloqueada usada de todas formas ---
    Row(tx_id=15, user_id=5, card_id=6,  merchant_id=1, amount=250.0, currency="USD", tx_date="2026-07-19 03:12:00", status="DECLINED"),
    Row(tx_id=16, user_id=5, card_id=6,  merchant_id=1, amount=250.0, currency="USD", tx_date="2026-07-19 03:13:00", status="DECLINED"),
    Row(tx_id=17, user_id=6, card_id=7,  merchant_id=3, amount=15.0,  currency="USD", tx_date="2026-07-19 09:30:00", status="APPROVED"),
    Row(tx_id=18, user_id=6, card_id=7,  merchant_id=4, amount=30.0,  currency="USD", tx_date="2026-07-20 17:00:00", status="APPROVED"),
    Row(tx_id=19, user_id=6, card_id=7,  merchant_id=6, amount=18.0,  currency="COP", tx_date="2026-07-21 20:15:00", status="APPROVED"),
    Row(tx_id=20, user_id=7, card_id=8,  merchant_id=2, amount=80.0,  currency="USD", tx_date="2026-07-19 11:00:00", status="APPROVED"),
    # --- monto atípico + hora rara ---
    Row(tx_id=21, user_id=7, card_id=8,  merchant_id=2, amount=3200.0, currency="USD", tx_date="2026-07-21 02:47:00", status="APPROVED"),
    Row(tx_id=22, user_id=8, card_id=9,  merchant_id=5, amount=45.0,  currency="COP", tx_date="2026-07-19 10:00:00", status="APPROVED"),
    # --- viaje imposible: tx23 y tx24 a 6 min, países distintos (ver transaction_details) ---
    Row(tx_id=23, user_id=8, card_id=9,  merchant_id=1, amount=60.0,  currency="USD", tx_date="2026-07-20 10:05:00", status="APPROVED"),
    Row(tx_id=24, user_id=8, card_id=9,  merchant_id=7, amount=700.0, currency="USD", tx_date="2026-07-20 10:11:00", status="APPROVED"),
    # --- card testing: micro-cobros y luego un cobro grande ---
    Row(tx_id=25, user_id=9, card_id=10, merchant_id=1, amount=1.0,   currency="USD", tx_date="2026-07-20 01:00:00", status="APPROVED"),
    Row(tx_id=26, user_id=9, card_id=10, merchant_id=1, amount=1.0,   currency="USD", tx_date="2026-07-20 01:01:00", status="APPROVED"),
    Row(tx_id=27, user_id=9, card_id=10, merchant_id=1, amount=1.0,   currency="USD", tx_date="2026-07-20 01:02:00", status="DECLINED"),
    Row(tx_id=28, user_id=9, card_id=10, merchant_id=1, amount=1.0,   currency="USD", tx_date="2026-07-20 01:02:30", status="DECLINED"),
    Row(tx_id=29, user_id=9, card_id=10, merchant_id=1, amount=850.0, currency="USD", tx_date="2026-07-20 01:05:00", status="APPROVED"),
    Row(tx_id=30, user_id=10, card_id=11, merchant_id=8, amount=33.0, currency="COP", tx_date="2026-07-19 13:00:00", status="APPROVED"),
    Row(tx_id=31, user_id=10, card_id=11, merchant_id=6, amount=21.0, currency="COP", tx_date="2026-07-20 20:30:00", status="APPROVED"),
    Row(tx_id=32, user_id=1, card_id=1,  merchant_id=1, amount=130.0, currency="USD", tx_date="2026-07-21 09:15:00", status="APPROVED"),
    Row(tx_id=33, user_id=2, card_id=3,  merchant_id=2, amount=60.0,  currency="USD", tx_date="2026-07-22 11:00:00", status="APPROVED"),
    Row(tx_id=34, user_id=4, card_id=5,  merchant_id=4, amount=25.0,  currency="USD", tx_date="2026-07-22 09:45:00", status="APPROVED"),
    Row(tx_id=35, user_id=6, card_id=7,  merchant_id=1, amount=40.0,  currency="USD", tx_date="2026-07-22 15:00:00", status="APPROVED"),
    Row(tx_id=36, user_id=7, card_id=8,  merchant_id=3, amount=15.0,  currency="USD", tx_date="2026-07-19 20:00:00", status="APPROVED"),
    Row(tx_id=37, user_id=10, card_id=11, merchant_id=2, amount=70.0, currency="USD", tx_date="2026-07-21 16:00:00", status="APPROVED"),
    Row(tx_id=38, user_id=5, card_id=6,  merchant_id=2, amount=90.0,  currency="USD", tx_date="2026-07-22 09:00:00", status="DECLINED"),
    Row(tx_id=39, user_id=8, card_id=9,  merchant_id=6, amount=33.0,  currency="COP", tx_date="2026-07-22 12:30:00", status="APPROVED"),
    Row(tx_id=40, user_id=3, card_id=4,  merchant_id=4, amount=20.0,  currency="USD", tx_date="2026-07-22 08:00:00", status="APPROVED"),
]

df_transactions = spark.createDataFrame(transactions_data)
df_transactions.createOrReplaceTempView("transactions")


# ==========================================
# TABLA 5: TRANSACTION_DETAILS
# ==========================================

details_data = [
    Row(tx_id=1,  source="web",    country="CO", city="Pasto",            ip_address="190.12.45.10", device_id="dev-101"),
    Row(tx_id=2,  source="pos",    country="CO", city="Pasto",            ip_address="190.12.45.10", device_id="dev-101"),
    Row(tx_id=3,  source="mobile", country="CO", city="Bogota",           ip_address="190.12.50.22", device_id="dev-102"),
    Row(tx_id=4,  source="web",    country="CO", city="Medellin",         ip_address="181.34.22.11", device_id="dev-201"),
    Row(tx_id=5,  source="mobile", country="CO", city="Medellin",         ip_address="181.34.22.11", device_id="dev-201"),
    Row(tx_id=6,  source="web",    country="CO", city="Medellin",         ip_address="181.34.22.15", device_id="dev-202"),
    Row(tx_id=7,  source="mobile", country="US", city="Miami",            ip_address="73.12.5.9",    device_id="dev-301"),
    Row(tx_id=8,  source="web",    country="US", city="Miami",            ip_address="45.88.12.4",   device_id="dev-777"),
    Row(tx_id=9,  source="web",    country="US", city="Miami",            ip_address="45.88.12.4",   device_id="dev-777"),
    Row(tx_id=10, source="web",    country="US", city="Miami",            ip_address="45.88.12.4",   device_id="dev-777"),
    Row(tx_id=11, source="web",    country="US", city="Miami",            ip_address="45.88.12.4",   device_id="dev-777"),
    Row(tx_id=12, source="web",    country="US", city="Miami",            ip_address="45.88.12.4",   device_id="dev-777"),
    Row(tx_id=13, source="mobile", country="CO", city="Cali",             ip_address="190.90.10.5",  device_id="dev-401"),
    Row(tx_id=14, source="pos",    country="CO", city="Cali",             ip_address="190.90.10.5",  device_id="dev-401"),
    Row(tx_id=15, source="web",    country="MX", city="Ciudad de Mexico", ip_address="201.14.8.3",   device_id="dev-501"),
    Row(tx_id=16, source="web",    country="MX", city="Ciudad de Mexico", ip_address="201.14.8.3",   device_id="dev-501"),
    Row(tx_id=17, source="mobile", country="CO", city="Bucaramanga",      ip_address="186.30.4.8",   device_id="dev-601"),
    Row(tx_id=18, source="pos",    country="CO", city="Bucaramanga",      ip_address="186.30.4.8",   device_id="dev-601"),
    Row(tx_id=19, source="web",    country="CO", city="Bucaramanga",      ip_address="186.30.4.9",   device_id="dev-601"),
    Row(tx_id=20, source="web",    country="US", city="Chicago",          ip_address="98.22.6.4",    device_id="dev-701"),
    Row(tx_id=21, source="web",    country="RU", city="Moscow",           ip_address="95.24.100.8",  device_id="dev-999"),
    Row(tx_id=22, source="mobile", country="CO", city="Pasto",            ip_address="190.12.45.20", device_id="dev-801"),
    Row(tx_id=23, source="web",    country="CO", city="Pasto",            ip_address="190.12.45.20", device_id="dev-801"),
    Row(tx_id=24, source="web",    country="RU", city="Moscow",           ip_address="95.24.101.9",  device_id="dev-802"),
    Row(tx_id=25, source="web",    country="MX", city="Guadalajara",      ip_address="189.30.1.2",   device_id="dev-901"),
    Row(tx_id=26, source="web",    country="MX", city="Guadalajara",      ip_address="189.30.1.2",   device_id="dev-901"),
    Row(tx_id=27, source="web",    country="MX", city="Guadalajara",      ip_address="189.30.1.2",   device_id="dev-901"),
    Row(tx_id=28, source="web",    country="MX", city="Guadalajara",      ip_address="189.30.1.2",   device_id="dev-901"),
    Row(tx_id=29, source="web",    country="MX", city="Guadalajara",      ip_address="189.30.1.3",   device_id="dev-902"),
    Row(tx_id=30, source="mobile", country="CO", city="Ibague",           ip_address="181.50.2.3",   device_id="dev-1001"),
    Row(tx_id=31, source="pos",    country="CO", city="Ibague",           ip_address="181.50.2.3",   device_id="dev-1001"),
    Row(tx_id=32, source="web",    country="CO", city="Pasto",            ip_address="190.12.45.10", device_id="dev-101"),
    Row(tx_id=33, source="mobile", country="CO", city="Medellin",         ip_address="181.34.22.11", device_id="dev-201"),
    Row(tx_id=34, source="web",    country="CO", city="Medellin",         ip_address="181.34.22.12", device_id="dev-201"),
    Row(tx_id=35, source="pos",    country="US", city="Miami",            ip_address="73.12.5.9",    device_id="dev-301"),
    Row(tx_id=36, source="mobile", country="US", city="Chicago",          ip_address="98.22.6.4",    device_id="dev-701"),
    Row(tx_id=37, source="web",    country="CO", city="Ibague",           ip_address="181.50.2.3",   device_id="dev-1001"),
    Row(tx_id=38, source="web",    country="MX", city="Ciudad de Mexico", ip_address="201.14.8.3",   device_id="dev-501"),
    Row(tx_id=39, source="mobile", country="CO", city="Pasto",            ip_address="190.12.45.20", device_id="dev-801"),
    Row(tx_id=40, source="mobile", country="US", city="Miami",            ip_address="73.12.5.9",    device_id="dev-301"),
]

df_details = spark.createDataFrame(details_data)
df_details.createOrReplaceTempView("transaction_details")


# ==========================================
# TABLA 6: FRAUD_LABELS
# ==========================================

fraud_ids = {
    8: "VELOCITY", 9: "VELOCITY", 10: "VELOCITY", 11: "VELOCITY", 12: "VELOCITY",
    15: "BLOCKED_CARD_ATTEMPT", 16: "BLOCKED_CARD_ATTEMPT",
    21: "AMOUNT_OUTLIER",
    24: "GEO_ANOMALY",
    25: "CARD_TESTING", 26: "CARD_TESTING", 27: "CARD_TESTING", 28: "CARD_TESTING",
    29: "POST_TESTING_PURCHASE",
    38: "BLOCKED_CARD_ATTEMPT",
}

fraud_labels_data = [
    Row(tx_id=i, is_fraud=(1 if i in fraud_ids else 0), fraud_type=fraud_ids.get(i))
    for i in range(1, 41)
]

df_fraud = spark.createDataFrame(fraud_labels_data)
df_fraud.createOrReplaceTempView("fraud_labels")

print("Tablas creadas: users, merchants, cards, transactions, transaction_details, fraud_labels")

Tablas creadas: users, merchants, cards, transactions, transaction_details, fraud_labels


In [4]:
spark.catalog.listTables()

[Table(name='cards', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='fraud_labels', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='merchants', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='transaction_details', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='transactions', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='users', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [5]:
# ¿Cuál es el monto total y el monto promedio de las transacciones fraudulentas (is_fraud=1) comparado con las no fraudulentas?

query = """
SELECT
    fl.is_fraud,
    SUM(tx.amount) AS total,
    AVG(tx.amount) AS avg
FROM transactions AS tx
LEFT JOIN fraud_labels AS fl
    ON fl.tx_id = tx.tx_id
GROUP BY
    fl.is_fraud
"""
rslt = spark.sql(query).show()

+--------+------+------+
|is_fraud| total|   avg|
+--------+------+------+
|       0|1104.8|44.192|
|       1|7443.0| 496.2|
+--------+------+------+



In [6]:
# ¿Cuántas transacciones tiene cada card_id? ¿Cuál tarjeta es la más usada?

query = """
SELECT
    cd.card_type,
    COUNT(tx.tx_id)
FROM transactions AS tx
LEFT JOIN cards AS cd
    ON cd.card_id = tx.card_id
GROUP BY
    cd.card_type
"""
rslt = spark.sql(query).show()


+---------+------------+
|card_type|count(tx_id)|
+---------+------------+
|   credit|          28|
|    debit|          12|
+---------+------------+



In [7]:
# ¿Cuántos usuarios tienen más de una tarjeta asociada en la tabla cards?

query = """
SELECT
    usr.full_name,
    cd.card_type,
    COUNT(cd.card_id)
FROM cards AS cd
LEFT JOIN users AS usr
    ON usr.user_id = cd.user_id
GROUP BY
    usr.full_name,
    cd.card_type
ORDER BY
    usr.full_name
"""
rslt = spark.sql(query).show()

+----------------+---------+--------------+
|       full_name|card_type|count(card_id)|
+----------------+---------+--------------+
|      Ana Torres|    debit|             1|
|Carlos Fernandez|   credit|             1|
|Carlos Fernandez|    debit|             1|
|    Diego Castro|   credit|             1|
|      John Smith|   credit|             1|
|     Julia Perez|    debit|             1|
|     Kevin Brown|   credit|             1|
|      Laura Diaz|    debit|             1|
|     Maria Lopez|   credit|             1|
|     Maria Lopez|    debit|             1|
|     Pedro Gomez|   credit|             1|
|   Sofia Ramirez|   credit|             1|
+----------------+---------+--------------+



In [8]:
# Lista las transacciones cuyo país de la transacción (transaction_details.country) es distinto al país registrado del usuario (users.country).

query =  """
SELECT *
FROM transactions AS tx
LEFT JOIN transaction_details AS td
    ON td.tx_id = tx.tx_id
LEFT JOIN users AS usr
    ON usr.user_id = tx.user_id
WHERE
    td.country != usr.country
"""
rslt = spark.sql(query).show()

+-----+-------+-------+-----------+------+--------+-------------------+--------+-----+------+-------+------+-----------+---------+-------+-------------+-------+-----------+----------+
|tx_id|user_id|card_id|merchant_id|amount|currency|            tx_date|  status|tx_id|source|country|  city| ip_address|device_id|user_id|    full_name|country|signup_date|risk_level|
+-----+-------+-------+-----------+------+--------+-------------------+--------+-----+------+-------+------+-----------+---------+-------+-------------+-------+-----------+----------+
|   35|      6|      7|          1|  40.0|     USD|2026-07-22 15:00:00|APPROVED|   35|   pos|     US| Miami|  73.12.5.9|  dev-301|      6|   Laura Diaz|     CO| 2021-12-01|       LOW|
|   21|      7|      8|          2|3200.0|     USD|2026-07-21 02:47:00|APPROVED|   21|   web|     RU|Moscow|95.24.100.8|  dev-999|      7|  Kevin Brown|     US| 2022-09-14|    MEDIUM|
|   24|      8|      9|          7| 700.0|     USD|2026-07-20 10:11:00|APPROVED|

In [9]:
# ¿Qué porcentaje del monto total transaccionado corresponde a transacciones marcadas como fraude?

query = """
SELECT
    SUM(tx.amount)
FROM transactions AS tx
LEFT JOIN fraud_labels AS fl
    ON fl.tx_id = tx.tx_id
"""
rslt = spark.sql(query).show()

+-----------+
|sum(amount)|
+-----------+
|     8547.8|
+-----------+



In [12]:
query = """
SELECT
    fl.is_fraud AS is_fraud,
    SUM(tx.amount) AS total,
    (
    SELECT
        SUM(tx.amount)
    FROM transactions AS tx
    ) AS total_tx
FROM transactions AS tx
LEFT JOIN fraud_labels AS fl
    ON fl.tx_id = tx.tx_id
GROUP BY
    fl.is_fraud
"""
rslt = spark.sql(query).show()

+--------+------+--------+
|is_fraud| total|total_tx|
+--------+------+--------+
|       0|1104.8|  8547.8|
|       1|7443.0|  8547.8|
+--------+------+--------+



In [ ]:
query = """
WITH table AS (
    SELECT
        fl.is_fraud AS is_fraud,
        SUM(tx.amount) AS total,
        (SELECT
            SUM(tx.amount)
        FROM transactions AS tx) AS total_tx
    FROM transactions AS tx
    LEFT JOIN fraud_labels AS fl
        ON fl.tx_id = tx.tx_id
    GROUP BY
        fl.is_fraud
)

SELECT
    *,
    ROUND((t.total * 100)/t.total_tx,2)
FROM table as t

"""
rslt = spark.sql(query).show()

+--------+------+--------+------------------------------------+
|is_fraud| total|total_tx|round(((total * 100) / total_tx), 2)|
+--------+------+--------+------------------------------------+
|       0|1104.8|  8547.8|                               12.92|
|       1|7443.0|  8547.8|                               87.08|
+--------+------+--------+------------------------------------+



In [14]:
# ¿Cuántas transacciones DECLINED tiene cada usuario? ¿Quién tiene más?

query = """
SELECT *
FROM transactions AS tx
LEFT JOIN users AS usr
    ON usr.user_id = tx.user_id
"""
rslt = spark.sql(query).show()

+-----+-------+-------+-----------+------+--------+-------------------+--------+-------+----------------+-------+-----------+----------+
|tx_id|user_id|card_id|merchant_id|amount|currency|            tx_date|  status|user_id|       full_name|country|signup_date|risk_level|
+-----+-------+-------+-----------+------+--------+-------------------+--------+-------+----------------+-------+-----------+----------+
|    1|      1|      1|          1| 120.0|     USD|2026-07-19 09:00:00|APPROVED|      1|Carlos Fernandez|     CO| 2021-03-15|       LOW|
|    2|      1|      1|          2|  45.5|     USD|2026-07-19 14:30:00|APPROVED|      1|Carlos Fernandez|     CO| 2021-03-15|       LOW|
|    3|      1|      2|          4|  22.0|     USD|2026-07-20 08:00:00|APPROVED|      1|Carlos Fernandez|     CO| 2021-03-15|       LOW|
|    4|      2|      3|          5|  95.0|     COP|2026-07-19 10:00:00|APPROVED|      2|     Maria Lopez|     CO| 2020-11-02|       LOW|
|    5|      2|      3|          6|  22.3

In [18]:
query = """
SELECT
    usr.full_name AS name,
    COUNT(CASE WHEN tx.status::string == 'DECLINED' THEN 1 END) AS status
FROM transactions AS tx
LEFT JOIN users AS usr
    ON usr.user_id = tx.user_id
GROUP BY
    name
ORDER BY
    status DESC
"""
rslt = spark.sql(query).show()

+----------------+------+
|            name|status|
+----------------+------+
|     Pedro Gomez|     3|
|    Diego Castro|     2|
|      John Smith|     1|
|   Sofia Ramirez|     0|
|Carlos Fernandez|     0|
|     Kevin Brown|     0|
|     Maria Lopez|     0|
|      Ana Torres|     0|
|      Laura Diaz|     0|
|     Julia Perez|     0|
+----------------+------+



In [22]:
# ¿Cuál es la fecha de la primera transacción fraudulenta registrada en todo el dataset?

query =  """
SELECT *
FROM transactions AS tx
LEFT JOIN fraud_labels AS fl
    ON fl.tx_id = tx.tx_id
WHERE
    fl.is_fraud == 1
ORDER BY
    tx.tx_date ASC
"""
rslt = spark.sql(query).show()

+-----+-------+-------+-----------+------+--------+-------------------+--------+-----+--------+--------------------+
|tx_id|user_id|card_id|merchant_id|amount|currency|            tx_date|  status|tx_id|is_fraud|          fraud_type|
+-----+-------+-------+-----------+------+--------+-------------------+--------+-----+--------+--------------------+
|   15|      5|      6|          1| 250.0|     USD|2026-07-19 03:12:00|DECLINED|   15|       1|BLOCKED_CARD_ATTEMPT|
|   16|      5|      6|          1| 250.0|     USD|2026-07-19 03:13:00|DECLINED|   16|       1|BLOCKED_CARD_ATTEMPT|
|   25|      9|     10|          1|   1.0|     USD|2026-07-20 01:00:00|APPROVED|   25|       1|        CARD_TESTING|
|   26|      9|     10|          1|   1.0|     USD|2026-07-20 01:01:00|APPROVED|   26|       1|        CARD_TESTING|
|   27|      9|     10|          1|   1.0|     USD|2026-07-20 01:02:00|DECLINED|   27|       1|        CARD_TESTING|
|   28|      9|     10|          1|   1.0|     USD|2026-07-20 01

In [27]:
# Calcula la tasa de fraude (%) agrupando por card_type (crédito vs débito).

query = """
SELECT *
FROM transactions AS tx
LEFT JOIN cards AS cd
    ON cd.card_id = tx.card_id
LEFT JOIN fraud_labels AS fl
    ON fl.tx_id = tx.tx_id
"""
rslt = spark.sql(query).show()

+-----+-------+-------+-----------+------+--------+-------------------+--------+-------+-------+---------+-----------+----------+-----+--------+--------------------+
|tx_id|user_id|card_id|merchant_id|amount|currency|            tx_date|  status|card_id|user_id|card_type|issued_date|is_blocked|tx_id|is_fraud|          fraud_type|
+-----+-------+-------+-----------+------+--------+-------------------+--------+-------+-------+---------+-----------+----------+-----+--------+--------------------+
|   19|      6|      7|          6|  18.0|     COP|2026-07-21 20:15:00|APPROVED|      7|      6|    debit| 2022-01-01|     false|   19|       0|                NULL|
|    7|      3|      4|          4|  18.0|     USD|2026-07-19 08:15:00|APPROVED|      4|      3|   credit| 2022-02-01|     false|    7|       0|                NULL|
|    6|      2|     12|          3|  15.0|     USD|2026-07-21 08:00:00|APPROVED|     12|      2|    debit| 2023-03-01|     false|    6|       0|                NULL|
|   

In [31]:
query = """
SELECT
    cd.card_type AS card_type,
    COUNT(CASE WHEN fl.is_fraud == 1 THEN 1 END) AS count_isfrd,
    COUNT(*) AS total_tx
FROM transactions AS tx
LEFT JOIN cards AS cd
    ON cd.card_id = tx.card_id
LEFT JOIN fraud_labels AS fl
    ON fl.tx_id = tx.tx_id
GROUP BY
    cd.card_type
"""
rslt = spark.sql(query).show()

+---------+-----------+--------+
|card_type|count_isfrd|total_tx|
+---------+-----------+--------+
|   credit|         15|      28|
|    debit|          0|      12|
+---------+-----------+--------+



In [33]:
query = """
WITH table AS (
    SELECT
        cd.card_type AS card_type,
        COUNT(CASE WHEN fl.is_fraud == 1 THEN 1 END) AS count_isfrd,
        COUNT(*) AS total_tx
    FROM transactions AS tx
    LEFT JOIN cards AS cd
        ON cd.card_id = tx.card_id
    LEFT JOIN fraud_labels AS fl
        ON fl.tx_id = tx.tx_id
    GROUP BY
        cd.card_type
)

SELECT
    *,
    ROUND(((t.count_isfrd * 100)/t.total_tx),2) AS fraud_rate
FROM table AS t
"""
rslt = spark.sql(query).show()

+---------+-----------+--------+----------+
|card_type|count_isfrd|total_tx|fraud_rate|
+---------+-----------+--------+----------+
|   credit|         15|      28|     53.57|
|    debit|          0|      12|       0.0|
+---------+-----------+--------+----------+

